# **TUTORIAL** 3: Run the plastics MFA

Dynamic material flow analysis (MFA) translates circular economy (CE)
performance parameters into annual plastics stocks and flows. `run_mfa(...)`
uses [REMIND-MFA](https://github.com/pik-piam/remind-mfa) 0.2.0 and its input
data 1.0.0, with the world divided into 12 regions, plastics divided into 12
families, and seven end-use groups.

## Before starting...

Complete [Tutorial 1](1_download_data.ipynb) in the same workspace, including the
material recipe written by `download_data(...)`.


In [ ]:
from pathlib import Path

import pandas as pd

from pytoce import run_mfa, set_workspace

## Select the workspace

Run `set_workspace(...)` before any later function call. It sets the active
workspace root for the current Python session. Use the directory from Tutorial 1.


In [ ]:
workspace = Path.home() / "Documents" / "pytoce_workspace"
workspace_report = set_workspace(workspace)

## Foreground scenarios

This example uses *2019frozen_SSP2demand*, selected with the scenario name
`frozen_2019_default_baseline`. It retains the REMIND-MFA default in-use stock
demand and lifetimes, while collection, treatment composition, and virgin
feedstock composition retain their regional 2019 values from 2020 onward.

The other packaged scenario, *REMIND-MFA_default_SSP2*, uses
`remind_mfa_default` and retains the REMIND-MFA default trajectories for all CE
performance parameters. Select that name in the calculation below to compare
the two pathways.

Runnable YAML files are stored in `data_raw/foreground_scenarios/scenarios/`.
The installed templates and parameter catalog explain how to create user-defined
foreground scenarios. The next cell shows their workspace paths.


In [ ]:
{
    "scenarios": workspace_report.foreground_scenarios,
    "templates": workspace_report.foreground_scenario_templates,
    "parameter_catalogue": workspace_report.parameter_catalogue,
}

## Calculate the foreground scenario

The first call calculates the scenario and writes its result tables. Later
calls reuse completed results. Use `refresh=True` to recompile and recalculate
a scenario after changing its inputs under the same name.


In [ ]:
foreground = "frozen_2019_default_baseline"
mfa_report = run_mfa(foreground_scenario=foreground)
mfa_report.as_dict()

## Read the result

Each report item identifies one foreground scenario and its result files. The
following cell reads the collection rate used by the MFA for EUR in 2050 from
`resolved_input_parameters`. The collection rate is expressed as a fraction.

The result directory also contains stocks, material flows, and Sankey tables.
`material_realization.parquet` reports treatment amounts, recovered resin and
monomers, virgin monomer use, and new polymer production. Its `flow_name`
column identifies each quantity and process stage. The guides beside the tables
describe their columns and units.


In [ ]:
mfa_result = next(
    result
    for result in mfa_report.results
    if result.foreground_scenario == "frozen_2019_default_baseline"
)
resolved_inputs = pd.read_parquet(
    mfa_result.resolved_input_parameters,
    filters=[
        ("year", "==", 2050),
        ("region", "==", "EUR"),
        ("parameter_id", "==", "collection_rate"),
    ],
)
resolved_inputs

User-defined foreground scenarios can modify in-use stock demand, mean
lifetimes, collection, treatment composition, and virgin feedstock composition.
Changes begin in 2020 while historical data through 2019 are retained. The
workspace guide at
`data_raw/foreground_scenarios/templates/README_foreground_scenarios.md`
explains the accepted inputs and scenario construction.

In each year, the stock remaining in use from previous cohorts is retained.
The new plastics inflow is calculated to meet the in-use stock target after
accounting for these surviving cohorts. End-of-life plastics follow the selected
collection and treatment parameters. Recovered materials contribute to supplying
new production, and virgin production supplies the remaining requirement.

The MFA checks cohort stocks, recovery flows, and mass balances before writing
the results. Plastics remaining in use at the end of 2100 are kept in stock.


## What to do next

Once the foreground scenario and backgrounds are complete, continue with
[Tutorial 4: Run the time-explicit prospective LCA](4_run_lca.ipynb).
